### Importing Libraries 

In [11]:
import websocket 
import json 
import pandas as pd
import time
import numpy as np
import torch 
from sklearn.preprocessing import StandardScaler
import torch.nn as nn

In [5]:
SYMBOL="btcusdt"
STREAM_URL=f"wss://stream.binance.com:9443/ws/{SYMBOL}@bookTicker"
MAX_TICKS=5000 # Number of rows to collect 

# Global list to store our processed data
tick_data=[]
start_time=time.time()


In [6]:
def on_message(ws,message):
    """Triggered every time Binance sends a new tick"""
    data= json.loads(message)

    # Extract raw level 2 Order Book Data 
    bid_price= float(data['b'])
    bid_qty=float(data['B'])
    ask_price= float(data['a'])
    ask_qty=float(data["A"])


    # --- Feature Engineering (On the fly) ---
    # 1.Mid-Price: The exact Center of the spread 
    mid_price= (bid_price + ask_price )/2.0

    # Order Book Imbalance (OBI): Pressure between buyers and sellers
    # Range is [-1 to 1]. Positive means more buying pressure, negative means selling pressure.
     
    obi = (bid_qty - ask_qty) / (bid_qty + ask_qty)

    tick_data.append({
        "timestamp":time.time(),
        "mid_price": mid_price,
        "obi": obi,
        "bid_qty": bid_qty,
        "ask_qty": ask_qty
        })
    
    # progress tracker
    if len(tick_data) % 500==0:
        print(f"collect {len(tick_data)}/{MAX_TICKS} ticks...")
    # stop condition
    if len(tick_data) >= MAX_TICKS:
        print(f"\n Target reached in {round(time.time() - start_time, 2 )} seconds. Closing connection...")

        ws.close()
    
def on_error(ws,error):
    print(f"Error: {error}")

def on_close(ws,close_status_code, close_msg):
    print("Connection closed.")

    df= pd.DataFrame(tick_data)
    df.to_csv("btc_hft_ticks.csv", index=False)
    print(f"Successfully saved {len(df)} rows to csv")
    print(df.head())

def on_open(ws):
    print(f"Connected to Binance {SYMBOL.upper()} stream. Waiting for ticks...")

if __name__ == "__main__":
    ws= websocket.WebSocketApp(
        STREAM_URL,
        on_open=on_open,
        on_message= on_message,
        on_error=on_error,
        on_close=on_close
    )
    ws.run_forever()

Connected to Binance BTCUSDT stream. Waiting for ticks...
collect 500/5000 ticks...
collect 1000/5000 ticks...
collect 1500/5000 ticks...
collect 2000/5000 ticks...
collect 2500/5000 ticks...
collect 3000/5000 ticks...
collect 3500/5000 ticks...
collect 4000/5000 ticks...
collect 4500/5000 ticks...
collect 5000/5000 ticks...

 Target reached in 71.27 seconds. Closing connection...
Connection closed.
Successfully saved 5000 rows to csv
      timestamp  mid_price       obi  bid_qty  ask_qty
0  1.782384e+09  61543.525 -0.846425  0.33913  4.07735
1  1.782384e+09  61543.525 -0.846415  0.33913  4.07707
2  1.782384e+09  61543.525 -0.846127  0.33982  4.07707
3  1.782384e+09  61543.525 -0.846089  0.33991  4.07707
4  1.782384e+09  61543.525 -0.846052  0.34000  4.07707


### Data Preprocessing and Target Labelling 

In [9]:
df=pd.read_csv("btc_hft_ticks.csv")

# Quant labeling with assistance of financial books to understand
LOOKAHEAD= 10

df["future_mid"]=df["mid_price"].shift(-LOOKAHEAD)
df["price_change"]= (df["future_mid"]-df["mid_price"])/ df["mid_price"]

# defining the volatility Threshold
# for ultra-fast ticks we are going to use the SDV of changes as a dynamic guide

threshold= df["price_change"].std() * 0.5

def assign_label(change):
    if pd.isna(change):
        return np.nan
    if change > threshold:
        return 2 # UP (BUY)
    elif change < -threshold: 
        return 0 # DOWN (SELL)
    else:
        return 1 # FLAT (HOLD)
    
df["target"]= df["price_change"].apply(assign_label)

df=df.dropna().reset_index(drop=True)

feature_cols=["obi","bid_qty","ask_qty"]
X_raw=df[feature_cols].values
Y_raw= df["target"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

print(f"Dataset summary after labeling:")
print(df["target"].value_counts(normalize=True))

Dataset summary after labeling:
target
1.0    0.934068
2.0    0.036673
0.0    0.029259
Name: proportion, dtype: float64


In [10]:
def create_hft_sequences(X_data, Y_data, seq_length=50):
    X_seq=[]
    Y_seq=[]

    for i in range(len(X_data)- seq_length):
        # Context window: past 'seq_length' ticks
        X_seq.append(X_data[i:i+seq_length])
        # Target: the label corresponding to the final tick in that context
        Y_seq.append(Y_data[i + seq_length - 1])

    return torch.tensor(np.array(X_seq), dtype=torch.float32), torch.tensor(np.array(Y_seq), dtype= torch.long)

SEQ_LENGTH = 50

X_tensor, Y_tensor = create_hft_sequences(X_scaled, Y_raw, seq_length=SEQ_LENGTH)

#To avoid look ahead leakage
split_idx= int(len(X_tensor) * 0.8)
X_train , X_test=X_tensor[:split_idx], X_tensor[split_idx:]
Y_train, Y_test= Y_tensor[:split_idx], Y_tensor[split_idx:]

print(f"X_train_shape: {X_train.shape}, Y_train_shape: {Y_train.shape}")


X_train_shape: torch.Size([3952, 50, 3]), Y_train_shape: torch.Size([3952])


In [12]:
class MambaHFTClassifier(nn.Module):
    def __init__(self,mamba_core_block, d_model, num_classes=3):
        super().__init__()

        self.mamba=mamba_core_block

        # Linear Layer to project features up if needed
        self.input_projection = nn.Linear(3, d_model)

        # Classifcation Head 
        # the final step's hidden state ad predicts [Down, Flat ,  Up]
        self.classification_head=nn.Linear(d_model, num_classes)

    def foward(self,x):
        # shape: (Batch, Sequence_length, Input_features) -> (B,50,3)
        x= self.input_projection(x) # converting to (B, 50, d_model)
        # passing sequence through the selective state space updates
        mamba_out= self.mamba(x) # (B, 50, d_model)

        final_step_state= mamba_out[:,-1,:] # shape: (B, d_model)

        # Output unnormalized logits for CrossEntropyLoss
        logits = self.classification_head(final_step_state) # Shape: (B,3)
        return logits 